# VIX Term Structure Strategy Demonstration

This notebook recreates `qc_projects/main_vix_term_structure.py` using a synthetic term-structure proxy:
- Front-month proxy: 10-day annualized realized volatility of SPY
- Back-month proxy: 30-day annualized realized volatility of SPY
- Term ratio = back_vol / front_vol
- Trade SPY long on backwardation and short on steep contango

## Step 1: Imports and Parameters

Parameters mirror the Lean defaults from the QC strategy.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

symbol = 'SPY'
start_date = '2020-01-01'
end_date = '2022-01-01'
front_window = 10
back_window = 30
contango_threshold = 1.05
backwardation_threshold = 0.95
long_position_size = 0.8
short_position_size = -0.5
warmup_period = 35

print(f'Symbol: {symbol} | Window: {start_date} → {end_date}')
print(f'Front/Back windows: {front_window}/{back_window}')
print(f'Thresholds: backwardation<{backwardation_threshold}, contango>{contango_threshold}')

## Step 2: Download Daily SPY Data

We use SPY as both the volatility proxy source and traded instrument.

In [ ]:
spy = yf.download(
    symbol,
    start=start_date,
    end=end_date,
    interval='1d',
    progress=False,
    multi_level_index=False,
    auto_adjust=False,
)
if spy.empty:
    raise RuntimeError('No OHLCV data returned for SPY')

spy = spy.dropna().copy()
spy.index = pd.to_datetime(spy.index).tz_localize(None)
spy.index.name = 'date'
spy.head()

## Step 3: Build Synthetic Term Structure

We annualize rolling standard deviation of daily returns to approximate short/long volatility legs.

In [ ]:
features = spy[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
features['returns'] = features['Close'].pct_change()
features['front_vol'] = features['returns'].rolling(front_window).std(ddof=0) * np.sqrt(252) * 100
features['back_vol'] = features['returns'].rolling(back_window).std(ddof=0) * np.sqrt(252) * 100
features['term_ratio'] = features['back_vol'] / features['front_vol']
features['term_ratio'] = features['term_ratio'].replace([np.inf, -np.inf], np.nan)
features = features.dropna(subset=['front_vol', 'back_vol', 'term_ratio']).copy()

if len(features) > warmup_period:
    features = features.iloc[warmup_period:].copy()

features[['Close', 'front_vol', 'back_vol', 'term_ratio']].head()

## Step 4: Replay Trading State Machine

- Enter long when ratio < 0.95 (backwardation)
- Enter short when ratio > 1.05 (steep contango)
- Exit long when ratio > 1.0
- Exit short when ratio < 1.0

In [ ]:
position = 0
entry_price = None
entry_time = None
entry_ratio = None
entry_size = 0.0

trades = []
signals = []

for ts, row in features.iterrows():
    ratio = float(row['term_ratio'])
    price = float(row['Close'])

    if position == 0:
        if ratio < backwardation_threshold:
            position = 1
            entry_price = price
            entry_time = ts
            entry_ratio = ratio
            entry_size = long_position_size
            signals.append({'timestamp': ts, 'type': 'entry', 'side': 'long', 'term_ratio': ratio, 'price': price})
        elif ratio > contango_threshold:
            position = -1
            entry_price = price
            entry_time = ts
            entry_ratio = ratio
            entry_size = short_position_size
            signals.append({'timestamp': ts, 'type': 'entry', 'side': 'short', 'term_ratio': ratio, 'price': price})

    elif position == 1:
        if ratio > 1.0:
            raw_ret = price / entry_price - 1.0
            ret_pct = raw_ret * abs(entry_size) * 100
            trades.append({
                'side': 'long',
                'entry_time': entry_time,
                'exit_time': ts,
                'entry_price': round(entry_price, 2),
                'exit_price': round(price, 2),
                'entry_ratio': round(entry_ratio, 4),
                'exit_ratio': round(ratio, 4),
                'return_pct': round(ret_pct, 3),
            })
            signals.append({'timestamp': ts, 'type': 'exit', 'side': 'long', 'term_ratio': ratio, 'price': price})
            position = 0

    elif position == -1:
        if ratio < 1.0:
            raw_ret = price / entry_price - 1.0
            ret_pct = (-raw_ret) * abs(entry_size) * 100
            trades.append({
                'side': 'short',
                'entry_time': entry_time,
                'exit_time': ts,
                'entry_price': round(entry_price, 2),
                'exit_price': round(price, 2),
                'entry_ratio': round(entry_ratio, 4),
                'exit_ratio': round(ratio, 4),
                'return_pct': round(ret_pct, 3),
            })
            signals.append({'timestamp': ts, 'type': 'exit', 'side': 'short', 'term_ratio': ratio, 'price': price})
            position = 0

trades_df = pd.DataFrame(trades)
signals_df = pd.DataFrame(signals)
print(f'Trades: {len(trades_df)} | Signals: {len(signals_df)}')
trades_df.head() if not trades_df.empty else 'No trades generated'

## Step 5: Aggregate Metrics

A quick performance summary before changing thresholds in QC.

In [ ]:
if trades_df.empty:
    summary = {
        'total_trades': 0,
        'win_rate_pct': 0.0,
        'avg_return_pct': 0.0,
        'median_return_pct': 0.0,
        'best_trade_pct': 0.0,
        'worst_trade_pct': 0.0,
    }
else:
    summary = {
        'total_trades': int(len(trades_df)),
        'win_rate_pct': round((trades_df['return_pct'] > 0).mean() * 100, 1),
        'avg_return_pct': round(float(trades_df['return_pct'].mean()), 3),
        'median_return_pct': round(float(trades_df['return_pct'].median()), 3),
        'best_trade_pct': round(float(trades_df['return_pct'].max()), 3),
        'worst_trade_pct': round(float(trades_df['return_pct'].min()), 3),
    }

summary

## Step 6: Visualize SPY, Volatility Curves, and Ratio Signals

We overlay entries/exits on the term-ratio panel and save an interactive chart.

In [ ]:
graphs_dir = Path('graphs')
graphs_dir.mkdir(exist_ok=True)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.40, 0.30, 0.30],
    subplot_titles=('SPY Price', 'Synthetic Volatility Proxies', 'Term Ratio Signals'),
)

fig.add_trace(go.Scatter(x=features.index, y=features['Close'], name='SPY Close', line=dict(color='#1f77b4')), row=1, col=1)
fig.add_trace(go.Scatter(x=features.index, y=features['front_vol'], name='Front Vol (10d)', line=dict(color='#ff7f0e')), row=2, col=1)
fig.add_trace(go.Scatter(x=features.index, y=features['back_vol'], name='Back Vol (30d)', line=dict(color='#2ca02c')), row=2, col=1)
fig.add_trace(go.Scatter(x=features.index, y=features['term_ratio'], name='Term Ratio', line=dict(color='#9467bd')), row=3, col=1)

fig.add_hline(y=contango_threshold, line=dict(color='#d62728', dash='dash'), row=3, col=1)
fig.add_hline(y=backwardation_threshold, line=dict(color='#2ca02c', dash='dash'), row=3, col=1)
fig.add_hline(y=1.0, line=dict(color='#7f7f7f', dash='dot'), row=3, col=1)

if not signals_df.empty:
    entries = signals_df[signals_df['type'] == 'entry']
    exits = signals_df[signals_df['type'] == 'exit']
    if not entries.empty:
        fig.add_trace(
            go.Scatter(
                x=entries['timestamp'],
                y=entries['term_ratio'],
                mode='markers',
                marker=dict(symbol='triangle-up', size=8, color='#2ca02c'),
                name='Entries',
            ),
            row=3, col=1
        )
    if not exits.empty:
        fig.add_trace(
            go.Scatter(
                x=exits['timestamp'],
                y=exits['term_ratio'],
                mode='markers',
                marker=dict(symbol='triangle-down', size=8, color='#d62728'),
                name='Exits',
            ),
            row=3, col=1
        )

fig.update_layout(height=850, title='VIX Term Structure Proxy Strategy')
output_path = graphs_dir / 'vix_term_structure_signals.html'
fig.write_html(output_path)
fig.show()
print(f'Saved interactive chart to {output_path}')

## Step 7: Trade Log and Recommendation

Review recent trades and generate a quick interpretation.

In [ ]:
if not trades_df.empty:
    display(trades_df.tail(12))

if summary['total_trades'] == 0:
    recommendation = 'HOLD - no strong term-structure extremes in this window.'
elif summary['win_rate_pct'] >= 55 and summary['avg_return_pct'] > 0:
    recommendation = 'BUY (strategy-ready) - volatility regime signals are constructive.'
elif summary['avg_return_pct'] <= 0:
    recommendation = 'AVOID/REFINE - adjust windows and thresholds.'
else:
    recommendation = 'NEUTRAL - mixed outcomes, continue calibration.'

print('Recommendation:', recommendation)

## Recap

- This demo replicates your QC VIX term-structure regime logic with a realized-volatility proxy.
- Tune `front_window`, `back_window`, and thresholds locally before pushing changes to Lean.
- Saved chart: `graphs/vix_term_structure_signals.html`.